# [3-1] 시계열 예측(Forecasting) AI 벤치마크
본 노트북은 과거의 교통량 데이터를 분석하여 **'미래의 교통 혼잡도'를 선제적으로 예측**하는 AI 모델을 검증합니다. 비전 AI(객체 탐지)가 눈으로 현재를 본다면, 시계열 AI(Forecasting)는 데이터의 패턴을 읽어 미래를 대비합니다.

### 🏆 참여 모델 (시계열 3대장)
1. **LSTM (Long Short-Term Memory):** 시계열 예측의 전통적인 절대 강자. 장기 기억력이 우수함.
2. **1D-CNN (Convolutional Neural Network):** 패턴 인식에 특화되어 속도가 매우 빠른 신흥 강자.
3. **GRU (Gated Recurrent Unit):** LSTM의 불필요한 연산을 줄여 속도와 효율성을 극대화한 최신 모델.

### 🎯 평가 목표
- 3개의 전혀 다른 아키텍처를 동시에 학습시켜, 교통 예측에 최적의(MSE 최소화, R² Score 최대화) 모델을 수학적으로 가려냅니다.
- 단순 수치를 넘어, **실제 교통량(Actual)**과 **AI의 예측값(Predicted)**을 시각적으로 오버레이하여 실무 투입 가능성을 증명합니다.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from dotenv import load_dotenv
from supabase import create_client, Client
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic' if os.name == 'nt' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"시계열 예측 벤치마크 환경 구성 완료 (Device: {device})")

### 1. 라이브 DB(Supabase) 연동 및 시뮬레이션 폴백(Fallback)
실제 `Total_Pipeline.ipynb`를 통해 적재된 Supabase 데이터베이스의 트래픽 로그를 우선적으로 불러옵니다. 단, 원활한 딥러닝 패턴 학습(쌍봉 패턴 등)을 위해 DB 데이터가 충분하지 않거나 연결되지 않을 경우, 정밀하게 설계된 가상 출퇴근 시뮬레이션 데이터를 사용하여 벤치마크를 진행합니다.

In [ ]:
def generate_traffic_data(days=14, points_per_day=24):
    total_points = days * points_per_day
    time = np.arange(total_points)
    base_traffic = 20
    
    daily_pattern = np.zeros(total_points)
    for i in range(total_points):
        hour = i % 24
        morning_peak = 50 * math.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
        evening_peak = 60 * math.exp(-0.5 * ((hour - 18) / 2.0) ** 2)
        daily_pattern[i] = morning_peak + evening_peak
        
    weekly_factor = np.ones(total_points)
    for i in range(total_points):
        day_of_week = (i // 24) % 7
        if day_of_week >= 5: weekly_factor[i] = 0.5
            
    noise = np.random.normal(0, 5, total_points)
    traffic = (base_traffic + daily_pattern) * weekly_factor + noise
    return np.maximum(traffic, 0)

load_dotenv(r'../.env')
url = os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_KEY')
ts_data = None
using_supabase = False
MIN_REQUIRED_HOURS = 200 # 최소 200시간(약 8일) 이상의 데이터가 있어야 학습 가능

if url and key:
    try:
        supabase: Client = create_client(url, key)
        print("Supabase DB 연결 성공. 실시간 트래픽 로그 조회를 시작합니다.")
        res = supabase.table('traffic_logs').select('*').order('created_at', desc=False).limit(20000).execute()
        df = pd.DataFrame(res.data)
        
        if len(df) > 0:
            df['created_at'] = pd.to_datetime(df['created_at'])
            df.set_index('created_at', inplace=True)
            if 'vehicle_count' in df.columns:
                df_ts = df.resample('1H').sum()[['vehicle_count']].fillna(0)
                temp_ts_data = df_ts['vehicle_count'].values
            else:
                df['count'] = 1
                df_ts = df.resample('1H').sum()[['count']].fillna(0)
                temp_ts_data = df_ts['count'].values
                
            if len(temp_ts_data) >= MIN_REQUIRED_HOURS:
                ts_data = temp_ts_data
                print(f"Supabase 라이브 데이터 {len(ts_data)}시간 분량 로드 완료")
                using_supabase = True
            else:
                print(f"⚠️ Supabase에 로그는 {len(df)}건 있으나, 수집된 시간 범위가 지나치게 짧습니다 ({len(temp_ts_data)}시간). 최소 {MIN_REQUIRED_HOURS}시간 필요. (Fallback 발동)")
        else:
            print("⚠️ Supabase에 적재된 데이터가 없습니다. (Fallback 발동)")
    except Exception as e:
        print(f"⚠️ Supabase 연동 에러 발생: {e} (Fallback 발동)")
else:
    print("⚠️ .env 파일에 Supabase 정보가 없어 연동을 건너뜁니다. (Fallback 발동)")

if not using_supabase:
    print("📊 안정적인 벤치마크를 위해 [가상 출퇴근(Double Peak) 교통량 데이터] 21일 치를 자동 생성하여 주입합니다.")
    ts_data = generate_traffic_data(days=21)

plt.figure(figsize=(15, 4))
plt.plot(ts_data[:24*7], color='#2C3E50', linewidth=2)
plt.title('학습용 교통량 데이터 (첫 1주일 패턴)', fontsize=15, fontweight='bold')
plt.xlabel('시간 (Hours)')
plt.ylabel('통과 차량 수')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 2. 딥러닝을 위한 시퀀스 데이터 전처리 (Windowing)
과거 `N`시간의 교통량을 보고, 다음 `1`시간의 교통량을 예측하도록 데이터를 조각(Window) 냅니다.

In [ ]:
seq_len = 12 # 과거 12시간을 보고 미래를 예측

scaler = MinMaxScaler()
ts_scaled = scaler.fit_transform(ts_data.reshape(-1, 1))

X, y = [], []
for i in range(len(ts_scaled) - seq_len):
    X.append(ts_scaled[i : i + seq_len])
    y.append(ts_scaled[i + seq_len])

X = np.array(X)
y = np.array(y)

# 80% Train, 20% Test
split = int(len(X) * 0.8)
X_train, y_train = torch.FloatTensor(X[:split]), torch.FloatTensor(y[:split])
X_test, y_test = torch.FloatTensor(X[split:]), torch.FloatTensor(y[split:])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

print(f"✅ 전처리 완료: 과거 {seq_len}시간 기반 학습 (Train: {len(X_train)}건, Test: {len(X_test)}건)")

### 3. 3대 최상위 시계열 모델(LSTM, 1D-CNN, GRU) 설계
서로 다른 아키텍처를 가진 3개의 인공지능 모델을 PyTorch로 직접 구현합니다.

In [ ]:
# 1. LSTM 모델 (표준형)
class TrafficLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# 2. 1D-CNN 모델 (속도/패턴 특화)
class TrafficCNN(nn.Module):
    def __init__(self, seq_len=12):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * seq_len, 1)
        
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = self.flatten(x)
        return self.fc(x)

# 3. GRU 모델 (효율성 우수)
class TrafficGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

print("LSTM, 1D-CNN, GRU 모델 아키텍처 로드 완료")

### 4. 3대 모델 동시 학습 및 평가 (Validation Loss 비교)
각 모델이 얼마나 빠르게 교통량 패턴을 이해하고 오차(Loss)를 줄여나가는지 검증합니다.

In [ ]:
epochs = 30
models = {
    '1D-CNN (패턴특화)': TrafficCNN(seq_len=seq_len).to(device),
    'LSTM (전통강자)': TrafficLSTM().to(device),
    'GRU (효율성우수)': TrafficGRU().to(device)
}

history = {name: [] for name in models.keys()}

print("🔄 3대 모델 인공지능 학습 시작...")
for name, model in models.items():
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_x.to(device))
            loss = criterion(pred, batch_y.to(device))
            loss.backward()
            optimizer.step()
            
        model.eval()
        with torch.no_grad():
            val_pred = model(X_test.to(device))
            val_loss = criterion(val_pred, y_test.to(device)).item()
            history[name].append(val_loss)
            
    print(f" - {name} 학습 완료 (최종 오차: {history[name][-1]:.4f})")

plt.figure(figsize=(10, 6))
colors = ['#E67E22', '#3498DB', '#9B59B6']
for i, (name, losses) in enumerate(history.items()):
    plt.plot(losses, label=name, linewidth=2, color=colors[i])

plt.title('📈 3대 모델별 검증 오차(Validation Loss) 하락 곡선', fontsize=15, fontweight='bold')
plt.xlabel('학습 반복 횟수 (Epochs)')
plt.ylabel('MSE Loss (낮을수록 좋음)')
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 5. [최종 결론] 정량적 평가 (R² Score) 및 예측 결과 시각화 증명
가장 높은 설명력(R² Score)을 나타내는 모델을 선정하고, 실제 테스트 데이터 위에 AI의 예측 선을 겹쳐 그려서(Overlay) 미래 예측이 얼마나 정확히 들어맞는지 시각적으로 검증합니다.

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        pred_scaled = model(X_test.to(device)).cpu().numpy()
    
    pred_real = scaler.inverse_transform(pred_scaled)
    y_real = scaler.inverse_transform(y_test.numpy())
    predictions[name] = pred_real.flatten()
    
    mse = mean_squared_error(y_real, pred_real)
    mae = mean_absolute_error(y_real, pred_real)
    r2 = r2_score(y_real, pred_real)
    results.append({'Model': name, 'MSE': mse, 'MAE': mae, 'R² Score (설명력)': r2})

df_res = pd.DataFrame(results).set_index('Model')
print("\n📊 1. [정량적 평가] 시계열 예측 모델 성능 표")
display(df_res)

print("\n📊 2. [가장 직관적인 평가] R² Score 그룹 막대그래프")
fig, ax = plt.subplots(figsize=(10, 5))
df_res['R² Score (설명력)'].plot(kind='bar', color=colors, ax=ax)
ax.set_title('🏆 모델별 R² Score (1.0에 가까울수록 높은 예측 정확도)', fontsize=15, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('R² Score')
plt.xticks(rotation=0, fontsize=12, fontweight='bold')
for i, v in enumerate(df_res['R² Score (설명력)'].values):
    ax.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

print("\n🔥 3. [최종 시각적 증명] 실제 트래픽 vs AI 예측 트래픽 오버레이")
best_model_name = df_res['R² Score (설명력)'].idxmax()
best_pred = predictions[best_model_name]

plt.figure(figsize=(15, 6))
plt.plot(y_real.flatten()[:120], label='실제 교통량 (Actual)', color='black', linewidth=3, alpha=0.7)
plt.plot(best_pred[:120], label=f'{best_model_name} 예측 (Predicted)', color='red', linestyle='--', linewidth=2)

plt.title(f'🔮 [미래 예측 검증] 1등 모델({best_model_name})의 트래픽 예측 정확도', fontsize=18, fontweight='bold')
plt.xlabel('미래 시간 (Hours)', fontsize=13)
plt.ylabel('통과 차량 수', fontsize=13)
plt.legend(fontsize=13)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\n💡 [최종 벤치마크 결과 해석]")
print(f"3개의 최상위 딥러닝 아키텍처를 테스트한 결과, {best_model_name} 모델이 가장 높은 R² Score를 달성했습니다.")
print("결과 그래프 분석 결과, 출퇴근 시간대의 교통량 급증(Peak) 구간에 대한 예측 모델의 추적 성능이 유의미한 수준임을 확인했습니다.")

### 6. [부록: 시나리오 검증] 돌발 사고(Anomaly) 탐지 시뮬레이터 🚨
최적화된 시계열 AI를 역이용하여, **'실제 교통량이 예측치(정상 범주)를 크게 벗어났을 때 이를 즉각적으로 탐지해 내는 관제 시스템'**의 시나리오를 시뮬레이션으로 검증합니다.

In [ ]:
# 정상 테스트 데이터 복사
scenario_actual = y_real.flatten()[:120].copy()
scenario_pred = best_pred[:120].copy()

# 🚨 가상의 돌발 사고 주입 (가장 트래픽이 높은 '출퇴근 피크 타임'에 대형 사고 발생으로 도로 마비)
# AI가 예측한 가장 막히는 시간(Peak)을 찾습니다.
peak_idx = np.argmax(scenario_pred[20:100]) + 20 

# 피크 타임에 급격히 통과 차량이 0대가 되는 '도로 전면 통제' 사고 시나리오
scenario_actual[peak_idx-2 : peak_idx+3] = [10, 0, 0, 0, 10]

# 오차 계산 및 이상치 임계값(Threshold) 설정
error = np.abs(scenario_actual - scenario_pred)
anomaly_threshold = 30 # 예측값과 실제값이 30대 이상 차이나면 사고로 간주
anomalies = np.where(error > anomaly_threshold)[0]

plt.figure(figsize=(15, 6))
plt.plot(scenario_actual, label='실제 교통량 (사고 발생)', color='black', linewidth=3)
plt.plot(scenario_pred, label=f'AI 정상 예측 ({best_model_name})', color='red', linestyle='--', linewidth=2, alpha=0.5)

# 이상치(사고) 지점에 붉은색 점 찍기 및 하이라이트
if len(anomalies) > 0:
    plt.scatter(anomalies, scenario_actual[anomalies], color='red', s=300, zorder=5, label='이상 탐지 (Anomaly)')
    plt.axvspan(anomalies[0]-3, anomalies[-1]+3, color='red', alpha=0.2, label='사고 발생 구간')

plt.title('🚨 [시뮬레이션] AI 관제 시스템의 돌발 사고(Anomaly) 자동 탐지', fontsize=18, fontweight='bold', color='darkred')
plt.xlabel('미래 시간 (Hours)', fontsize=13)
plt.ylabel('통과 차량 수', fontsize=13)
plt.legend(fontsize=13, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\n💡 [이상치 탐지 시스템 해석]")
if len(anomalies) > 0:
    print(f"⚠️ 시스템 경고: 시간대 {anomalies[0]} (출퇴근 피크타임) 부근에서 AI의 정상 예측치와 실제 트래픽 간에 극심한 오차가 발생했습니다.")
    print("⚠️ 도로 전면 통제로 인한 차량 흐름 '0' 상태를 확인했습니다.")
    print("✅ AI 관제 시스템이 이를 즉각 '돌발 사고'로 규정하고 관리자에게 알람을 전송합니다.")
    print("✅ 이처럼 시계열 예측 AI는 단순히 미래를 맞추는 것을 넘어, '현재의 이상 징후'를 실시간으로 잡아내는 효과적인 방어 체계가 됩니다.")
else:
    print("이상치가 탐지되지 않았습니다. Threshold를 조절해 보세요.")